# 🤖 Task 5: Auto Tagging Support Tickets Using LLM

### Project Overview
Customer service teams handle a massive volume of daily queries. Manually reading and routing these tickets is highly inefficient.
This notebook demonstrates an automated NLP pipeline that leverages a Large Language Model (LLM) to read customer queries and instantly predict the **top 3 most applicable categories**.

### Dataset Used
We utilize the **Banking77** dataset from Hugging Face, which contains various banking-related customer service requests (e.g., card issues, top-up failures, etc.).

### Methodology
We will implement and compare two prompting strategies using a pre-trained Transformer model:
1. **Zero-Shot Classification:** Categorizing tickets purely based on label semantics without prior examples.
2. **Few-Shot Learning:** Injecting context by providing the model with a few labeled examples before asking it to classify a new ticket.

In [1]:
# Import necessary libraries for data manipulation and modeling
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset
from transformers import pipeline
from tqdm import tqdm

### 1. Data Acquisition
Fetching the Banking77 dataset via the Hugging Face Hub.

In [2]:
# Load the dataset
hf_dataset = load_dataset("banking77")
hf_dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/298k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/93.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})

### 2. Data Preprocessing
Converting the raw dataset into a Pandas DataFrame for easier inspection and manipulation.

In [3]:
# Convert to DataFrame
support_df = pd.DataFrame(hf_dataset["train"])
support_df.head()

,text,label
0,I am still waiting on my card?,11
1,What can I do if my card still hasn't arrived ...,11
2,I have been waiting over a week. Is the card s...,11
3,Can I track my card while it is in the process...,11
4,"How do I know if I will get my card, or if it ...",11


### Feature Selection
For this pipeline, we only need the customer's text. We will isolate it and rename the column to represent a support ticket.

In [4]:
# Keep only the text column and rename it for clarity
support_df = support_df[["text"]]
support_df = support_df.rename(columns={"text": "customer_query"})
support_df.head()

,customer_query
0,I am still waiting on my card?
1,What can I do if my card still hasn't arrived ...
2,I have been waiting over a week. Is the card s...
3,Can I track my card while it is in the process...
4,"How do I know if I will get my card, or if it ..."


# Check original dimensions
support_df.shape

In [5]:
# Check original dimensions
support_df.shape

(10003, 1)

### Downsampling for Prototyping
To speed up the inference time during this experiment, we will take a random sample of 100 tickets.

In [6]:
# Sample 100 records
support_df = support_df.sample(100, random_state=42)
support_df.head()

,customer_query
6883,Is it possible for me to change my PIN number?
5836,I'm not sure why my card didn't work
8601,I don't think my top up worked
2545,Can you explain why my payment was charged a fee?
8697,How long does a transfer from a UK account tak...


### 3. Defining the Target Categories
Here we define the predefined tags that the support system uses to route tickets.

In [7]:
# Define the business categories
ticket_categories = [
    "Account Access Problem",
    "Payment or Billing Issue",
    "Card Problem",
    "Transaction Issue",
    "Technical App Problem",
    "Refund Request",
    "Fraud or Security Issue",
    "General Inquiry"
]
ticket_categories

['Account Access Problem',
 'Payment or Billing Issue',
 'Card Problem',
 'Transaction Issue',
 'Technical App Problem',
 'Refund Request',
 'Fraud or Security Issue',
 'General Inquiry']

### 4. Zero-Shot Classification Pipeline
We instantiate a pre-trained sequence classifier.
*(Note: Zero-shot requires no fine-tuning; the model relies on its pre-existing language understanding to map text to the provided labels).*

In [9]:
# Initialize the zero-shot classifier directly from Hugging Face Hub
zs_classifier = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli"
)

config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

### Testing Zero-Shot on a Single Query

In [10]:
# Extract one query to test
sample_query = support_df["customer_query"].iloc[0]

# Run prediction
zs_result = zs_classifier(sample_query, ticket_categories)
zs_result

{'sequence': 'Is it possible for me to change my PIN number?',
 'labels': ['Transaction Issue',
  'Payment or Billing Issue',
  'Card Problem',
  'General Inquiry',
  'Account Access Problem',
  'Fraud or Security Issue',
  'Technical App Problem',
  'Refund Request'],
 'scores': [0.21937839686870575,
  0.2091541290283203,
  0.1406032294034958,
  0.11425554007291794,
  0.11197451502084732,
  0.07972835004329681,
  0.0705871507525444,
  0.05431867390871048]}

### Isolating Top 3 Predictions

In [11]:
# Slice the top 3 highest probability labels
top_three_preds = zs_result["labels"][:3]
top_three_preds

['Transaction Issue', 'Payment or Billing Issue', 'Card Problem']

### Executing Zero-Shot Across the Dataset

In [12]:
zero_shot_output = []

# Iterate through all customer queries
for query in support_df["customer_query"]:
    prediction = zs_classifier(query, ticket_categories)
    top_3_tags = prediction["labels"][:3]
    zero_shot_output.append(top_3_tags)

# Attach results to the dataframe
support_df["zero_shot_preds"] = zero_shot_output
support_df.head()

,customer_query,zero_shot_preds
6883,Is it possible for me to change my PIN number?,"[Transaction Issue, Payment or Billing Issue, ..."
5836,I'm not sure why my card didn't work,"[Card Problem, Transaction Issue, Refund Request]"
8601,I don't think my top up worked,"[General Inquiry, Transaction Issue, Payment o..."
2545,Can you explain why my payment was charged a fee?,"[Transaction Issue, Refund Request, Account Ac..."
8697,How long does a transfer from a UK account tak...,"[Transaction Issue, General Inquiry, Account A..."


### 5. Few-Shot Learning Implementation
To enhance accuracy, we will construct a prompt that includes a few labeled examples. This acts as a guide for the LLM to understand the mapping logic better.

In [13]:
# Define context examples
shot_examples = """
Ticket: I forgot my password and cannot login
Tag: Account Access Problem

Ticket: My card payment failed
Tag: Card Problem

Ticket: I was charged twice for my order
Tag: Payment or Billing Issue
"""

### Building the Prompt Generator

In [14]:
def generate_few_shot_prompt(query_text):
    prompt_template = f"""
    You are an intelligent support ticket routing assistant.

    Allowed Categories:
    {ticket_categories}

    Reference Examples:
    {shot_examples}

    Task: Analyze the following user query and output the top 3 matching categories.

    Ticket: {query_text}
    """
    return prompt_template

### Validating the Prompt Structure

In [15]:
# Test the prompt format
sample_query = support_df["customer_query"].iloc[0]
generated_prompt = generate_few_shot_prompt(sample_query)
print(generated_prompt)


    You are an intelligent support ticket routing assistant.
    
    Allowed Categories:
    ['Account Access Problem', 'Payment or Billing Issue', 'Card Problem', 'Transaction Issue', 'Technical App Problem', 'Refund Request', 'Fraud or Security Issue', 'General Inquiry']
    
    Reference Examples:
    
Ticket: I forgot my password and cannot login
Tag: Account Access Problem

Ticket: My card payment failed
Tag: Card Problem

Ticket: I was charged twice for my order
Tag: Payment or Billing Issue

    
    Task: Analyze the following user query and output the top 3 matching categories.
    
    Ticket: Is it possible for me to change my PIN number?
    


### Executing Few-Shot Across the Dataset

In [16]:
few_shot_output = []

for query in support_df["customer_query"]:
    # Generate the contextual prompt
    structured_prompt = generate_few_shot_prompt(query)

    # Run classification
    prediction = zs_classifier(structured_prompt, ticket_categories)

    # Extract top 3
    top_3_tags = prediction["labels"][:3]
    few_shot_output.append(top_3_tags)

# Attach few-shot results to the dataframe
support_df["few_shot_preds"] = few_shot_output
support_df.head()

,customer_query,zero_shot_preds,few_shot_preds
6883,Is it possible for me to change my PIN number?,"[Transaction Issue, Payment or Billing Issue, ...","[Card Problem, General Inquiry, Transaction Is..."
5836,I'm not sure why my card didn't work,"[Card Problem, Transaction Issue, Refund Request]","[Payment or Billing Issue, Fraud or Security I..."
8601,I don't think my top up worked,"[General Inquiry, Transaction Issue, Payment o...","[Payment or Billing Issue, Card Problem, Fraud..."
2545,Can you explain why my payment was charged a fee?,"[Transaction Issue, Refund Request, Account Ac...","[Refund Request, Payment or Billing Issue, Tra..."
8697,How long does a transfer from a UK account tak...,"[Transaction Issue, General Inquiry, Account A...","[Payment or Billing Issue, Fraud or Security I..."


### 6. Strategy Comparison
Reviewing the predictions to see how the injected context (Few-Shot) altered the model's routing decisions compared to the baseline (Zero-Shot).

In [17]:
# Create a cleaner view for comparison
final_comparison = support_df[["customer_query", "zero_shot_preds", "few_shot_preds"]]
final_comparison.head(10)

,customer_query,zero_shot_preds,few_shot_preds
6883,Is it possible for me to change my PIN number?,"[Transaction Issue, Payment or Billing Issue, ...","[Card Problem, General Inquiry, Transaction Is..."
5836,I'm not sure why my card didn't work,"[Card Problem, Transaction Issue, Refund Request]","[Payment or Billing Issue, Fraud or Security I..."
8601,I don't think my top up worked,"[General Inquiry, Transaction Issue, Payment o...","[Payment or Billing Issue, Card Problem, Fraud..."
2545,Can you explain why my payment was charged a fee?,"[Transaction Issue, Refund Request, Account Ac...","[Refund Request, Payment or Billing Issue, Tra..."
8697,How long does a transfer from a UK account tak...,"[Transaction Issue, General Inquiry, Account A...","[Payment or Billing Issue, Fraud or Security I..."
5573,Why am I getting declines when trying to make ...,"[Transaction Issue, General Inquiry, Account A...","[Card Problem, Transaction Issue, General Inqu..."
576,What is the $1 transaction on my account?,"[Transaction Issue, Account Access Problem, Fr...","[General Inquiry, Card Problem, Transaction Is..."
6832,It looks like my card payment was sent back.,"[Card Problem, Transaction Issue, Refund Request]","[Card Problem, Transaction Issue, Payment or B..."
7111,Why am I unable to transfer money when I was a...,"[Transaction Issue, Account Access Problem, Re...","[Card Problem, Refund Request, Transaction Issue]"
439,What if there is an error on the exchange rate?,"[Transaction Issue, Payment or Billing Issue, ...","[Card Problem, Transaction Issue, General Inqu..."


### 7. Project Conclusion
In this pipeline, we successfully engineered an automated ticketing system using an LLM.

By experimenting with the Banking77 dataset, we demonstrated that:
1. **Zero-Shot Learning** provides a strong, immediate baseline for categorizing text without requiring massive labeled datasets.
2. **Few-Shot Prompting** effectively steers the model's focus, helping it recognize subtle contexts (like distinguishing between a generic card issue and a specific transaction issue) by supplying just a few reference examples.

This implementation highlights the immense potential of LLMs in optimizing customer service operations, reducing manual triage time, and ensuring faster response rates.